# Medical Image Annotation — Evaluation Framework

**Context.** When evaluating a classifier for medical-image annotation, aggregate accuracy hides the errors that matter. A model with 96% accuracy can still misclassify the one minority class a clinician actually needs. This notebook walks through the evaluation framework I applied in the Duke AI PM program capstone, using `scikit-learn`'s digits dataset as a public stand-in for a multi-class medical classifier.

**What this notebook produces.**
1. Per-class precision, recall, and F1 — not just headline accuracy.
2. A normalized confusion matrix to surface systematic misclassifications.
3. One-vs-rest ROC curves per class with AUC.
4. A threshold-sensitivity sweep showing the precision/recall tradeoff.
5. Product-management implications — what a PM does with these numbers.

**Why a stand-in dataset.** The actual capstone used a clinical image corpus under Duke's data-use agreement. The framework below is dataset-agnostic: swap the dataset loader and everything downstream works identically.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load dataset and train baseline

The digits dataset has 10 classes, ~180 samples per class — small enough to train in seconds, large enough to expose real evaluation patterns.

In [ ]:
data = load_digits()
X, y = data.data, data.target
class_names = [str(c) for c in data.target_names]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

print(f"Trained on {len(X_train)} samples, evaluating on {len(X_test)}.")

## 2. Per-class precision, recall, and F1

Headline accuracy compresses signal. A per-class report shows which classes the model handles well and which are under-performing — the classes a PM needs to either improve, deprioritize, or route to human review.

In [ ]:
report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).T.round(3)
print(report_df.loc[class_names + ["accuracy", "macro avg", "weighted avg"]].to_string())

## 3. Normalized confusion matrix

Normalizing by row (true label) makes the diagonal directly readable as recall per class, and off-diagonal cells as the specific misclassifications — e.g., 'class 9 is most often confused with class 5'.

In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(len(class_names)), class_names)
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Normalized confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if cm[i, j] > 0.01:
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i,j] > 0.5 else "black", fontsize=9)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4. One-vs-rest ROC curves

ROC + AUC per class answers: for each class, how well does the model discriminate 'this class' from 'everything else'? Useful when the PM wants to pick a probability threshold for a specific class (e.g., flag malignant tissue aggressively, benign conservatively).

In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(len(class_names))))

fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"Class {name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-rest ROC curves")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Threshold sensitivity — precision vs recall

The default 0.5 threshold is almost never the right one for a clinical product. This cell picks the worst-performing class from section 2 and sweeps thresholds to expose the precision/recall tradeoff the PM actually needs to price.

In [ ]:
worst_class_idx = int(np.argmin([report[c]["f1-score"] for c in class_names]))
worst_class = class_names[worst_class_idx]

precisions, recalls, thresholds = precision_recall_curve(
    y_test_bin[:, worst_class_idx], y_proba[:, worst_class_idx]
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precisions[:-1], label="Precision", color="#6f42c1")
ax.plot(thresholds, recalls[:-1], label="Recall", color="#2ea043")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title(f"Precision/Recall tradeoff — worst-F1 class: {worst_class}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Product-management implications

| Metric | What the PM does with it |
|---|---|
| **Per-class F1** | Identify under-performing classes → decide: improve with more labeled data, route to human review, or deprioritize in scope. |
| **Confusion pairs** | Off-diagonal cells show systematic confusion (e.g., 9↔4). These are the cases to surface to domain experts for disambiguation guidelines. |
| **ROC/AUC per class** | Pick per-class thresholds aligned to clinical cost. High recall for high-risk classes (no false negatives), high precision for low-risk. |
| **Threshold curve** | Communicate the tradeoff to stakeholders in their language: 'at 0.7 threshold we catch 85% of class X but 30% of flags are wrong — is that acceptable?' |

This evaluation framework was the backbone of the build-vs-buy decision in the capstone — quantifying not just whether a model was good enough, but *where* it was good enough and where a human-in-the-loop step was required.